In [ ]:
import pennylane as qml
import numpy as np
from pennylane.kernels import target_alignment
from sklearn.preprocessing import MinMaxScaler

N_QUBITS = 10
dev = qml.device("default.qubit", wires=N_QUBITS)


def ansatz_sem_hadamard(x, wires):
    qml.AngleEmbedding(x, wires=wires, rotation="X")
    qml.BasicEntanglerLayers(
        weights=np.zeros((1, len(wires))),
        wires=wires,
        rotation=qml.RZ
    )
    qml.AngleEmbedding(x, wires=wires, rotation="Y")


def ansatz_com_hadamard(x, wires):
    for w in wires:
        qml.Hadamard(wires=w)
    qml.AngleEmbedding(x, wires=wires, rotation="X")
    qml.BasicEntanglerLayers(
        weights=np.zeros((1, len(wires))),
        wires=wires,
        rotation=qml.RZ
    )
    qml.AngleEmbedding(x, wires=wires, rotation="Y")


def ansatz_strong_sem_hadamard(x, wires):
    qml.AngleEmbedding(x, wires=wires, rotation="X")
    qml.StronglyEntanglingLayers(
        weights=np.zeros((1, len(wires), 3)),
        wires=wires
    )
    qml.AngleEmbedding(x, wires=wires, rotation="Y")


def ansatz_strong_com_hadamard(x, wires):
    for w in wires:
        qml.Hadamard(wires=w)
    qml.AngleEmbedding(x, wires=wires, rotation="X")
    qml.StronglyEntanglingLayers(
        weights=np.zeros((1, len(wires), 3)),
        wires=wires
    )
    qml.AngleEmbedding(x, wires=wires, rotation="Y")


@qml.qnode(dev)
def kernel_baseline(x1, x2):
    qml.AngleEmbedding(x1, wires=range(N_QUBITS), rotation="X")
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(N_QUBITS), rotation="X")
    return qml.expval(qml.Projector([0] * N_QUBITS, wires=range(N_QUBITS)))


@qml.qnode(dev)
def kernel_sem_hadamard(x1, x2):
    ansatz_sem_hadamard(x1, wires=range(N_QUBITS))
    qml.adjoint(ansatz_sem_hadamard)(x2, wires=range(N_QUBITS))
    return qml.expval(qml.Projector([0] * N_QUBITS, wires=range(N_QUBITS)))


@qml.qnode(dev)
def kernel_com_hadamard(x1, x2):
    ansatz_com_hadamard(x1, wires=range(N_QUBITS))
    qml.adjoint(ansatz_com_hadamard)(x2, wires=range(N_QUBITS))
    return qml.expval(qml.Projector([0] * N_QUBITS, wires=range(N_QUBITS)))


@qml.qnode(dev)
def kernel_strong_sem_hadamard(x1, x2):
    ansatz_strong_sem_hadamard(x1, wires=range(N_QUBITS))
    qml.adjoint(ansatz_strong_sem_hadamard)(x2, wires=range(N_QUBITS))
    return qml.expval(qml.Projector([0] * N_QUBITS, wires=range(N_QUBITS)))


@qml.qnode(dev)
def kernel_strong_com_hadamard(x1, x2):
    ansatz_strong_com_hadamard(x1, wires=range(N_QUBITS))
    qml.adjoint(ansatz_strong_com_hadamard)(x2, wires=range(N_QUBITS))
    return qml.expval(qml.Projector([0] * N_QUBITS, wires=range(N_QUBITS)))


scaler = MinMaxScaler(feature_range=(0, np.pi))
X_tr = scaler.fit_transform(X_train[features].to_numpy())

kernels = {
    "Baseline (AngleX sem entanglement)": kernel_baseline,
    "Basic sem Hadamard":                 kernel_sem_hadamard,
    "Basic com Hadamard":                 kernel_com_hadamard,
    "Strong sem Hadamard":                kernel_strong_sem_hadamard,
    "Strong com Hadamard":                kernel_strong_com_hadamard,
}

alignments = {}
for name, k in kernels.items():
    alignment = target_alignment(X_tr, y_train, k, assume_normalized_kernel=True)
    alignments[name] = alignment
    print(f"{name}: {alignment:.4f}")

best = max(alignments, key=alignments.get)
print(f"\nMelhor kernel: {best} (alignment={alignments[best]:.4f})")